# Color Calibration

Color calibration standardizes color appearance across recordings made under different lighting
conditions, with different cameras, or across sessions. Two scenarios arise in ecological
imaging: correcting **still images** (morphometric photographs, drone stills) and correcting
**video recordings** (behavioral experiments, continuous monitoring) — the focus of
*An Imaging Primer for Ecologists* (Francisco et al., in prep.).

**What you need:** a color reference target — a neutral gray card (white balance only) or a
24-patch color checker (full correction). Hold it in front of the camera under the same lighting
used for the experiment at the start of each session.

> Set your camera to a **fixed white balance** before recording. Adaptive white balance changes
> the color balance frame to frame, making post-hoc correction impossible. GoPros default to
> adaptive — disable this before each session.

This notebook has two parts:

1. **Software walkthroughs** (below) — step-by-step GUI guides for Lightroom, GIMP,
   DaVinci Resolve, Final Cut Pro, and ffmpeg. Screenshots are placeholders to be filled in.
2. **Python / OpenCV** — a runnable implementation that computes white-balance gains and a
   full Color Correction Matrix, then applies them to a video.


## Still images — Lightroom

(Current Lightroom — the cloud-based app. For the older Lightroom Classic the steps are
equivalent but live in the **Develop** module instead of the **Edit** panel.)

1. Import all images from the session.
2. Select the reference-target image and open the **Edit** panel (press `E`, or click the
   sliders icon at the top right).
3. Open the **Color** section. Click the **White Balance Selector** eyedropper, then click the
   neutral gray patch in the image.
4. For finer control: open **Light -> Curve**, switch to the Red, Green, and Blue point
   curves in turn, and pull each toward the reference values (see the GIMP table below).
5. Right-click the corrected photo -> **Copy Edit Settings** (`Ctrl/Cmd+C`) -> check
   **White Balance** -> **Copy**.
6. Select all other images from the session -> right-click -> **Paste Edit Settings**
   (`Ctrl/Cmd+V`).
7. Export with a consistent color space: **File -> Export** (or right-click -> **Export**),
   open **More Options**, and set the color space — sRGB for display, AdobeRGB or
   ProPhoto RGB for quantitative analysis.

![Lightroom: White Balance Selector on the gray patch](images/lightroom_01_white_balance.png)
*Screenshot placeholder — Lightroom Edit panel, Color section, White Balance Selector active.*

![Lightroom: Copy/Paste edit settings across the session](images/lightroom_02_paste_settings.png)
*Screenshot placeholder — pasting the corrected white balance to the rest of the session.*


## Still images — GIMP

The neutral patches on a 24-patch color checker have these sRGB reference values:

| Patch | Name        | R, G, B       |
| ----- | ----------- | ------------- |
| 19    | White       | 243, 243, 242 |
| 22    | Neutral 5   | 122, 122, 121 |
| 24    | Black       |  52,  52,  52 |

1. Open the reference image: `File -> Open`.
2. Open Curves: `Colors -> Curves`.
3. Switch to the **Red** channel. Hover over the neutral gray patch to read its current value.
   Add a control point at that input value and drag the output to match the reference
   (e.g. if Red reads 140 where it should be 122, pull that point to 122).
   Repeat for **Green** and **Blue**.
4. Click **OK**. Save the curve: folder icon in the Curves dialog -> **Export Settings**.
5. To batch-apply to a folder of images, use Script-Fu (`Filters -> Script-Fu -> Console`):

```scheme
; Adjust the control-point arrays to match your measured values.
(let* ((filelist (cadr (file-glob "/path/to/images/*.tif" 1))))
  (for-each
    (lambda (filename)
      (let* ((image    (car (gimp-file-load RUN-NONINTERACTIVE filename filename)))
             (drawable (car (gimp-image-get-active-drawable image))))
        (gimp-curves-spline drawable HISTOGRAM-RED   10 #(0 0 64 60 128 122 255 255 255 255))
        (gimp-curves-spline drawable HISTOGRAM-GREEN 10 #(0 0 64 64 128 128 255 255 255 255))
        (gimp-curves-spline drawable HISTOGRAM-BLUE  10 #(0 0 64 68 128 130 255 255 255 255))
        (file-tiff-save RUN-NONINTERACTIVE image drawable
                        (string-append "/path/to/output/" (basename filename))
                        (basename filename) 0)
        (gimp-image-delete image)))
    filelist))
```

![GIMP: Curves dialog, Red channel control point](images/gimp_01_curves_red.png)
*Screenshot placeholder — GIMP Curves dialog, dragging the Red channel point onto the reference value.*


## Video — general approach

1. **Film the reference target** in the first few seconds of each recording, under the same
   lighting as the experiment.
2. **Extract the reference frame** from the video.
3. **Compute the correction** from that frame.
4. **Apply to the whole clip** — DaVinci and Final Cut apply the grade to the entire clip
   automatically; ffmpeg and the Python code below process the video frame by frame.

To extract the reference frame with ffmpeg:

```bash
# Extract frame at timestamp 00:00:02 (adjust to where the target appears)
ffmpeg -ss 00:00:02 -i input.mp4 -frames:v 1 reference_frame.png
```


## Video — DaVinci Resolve

DaVinci Resolve is one of the most powerful options for video color correction. The Color page
applies corrections to the entire clip without re-encoding, and grades can be copied to all
clips in one step.

1. Create a new project, import the video, and place it on the timeline (Edit page).
2. Switch to the **Color** page.
3. Scrub to the frame where the reference target is visible.
4. In the **Color Wheels** panel, click the **Auto Balance** eyedropper icon and click on the
   neutral gray patch in the viewer. DaVinci rescales the lift/gamma/gain for all three
   channels so the patch reads neutral.
5. For finer control: open **Curves** (`Shift+S`) -> right-click in the viewer ->
   **Set White Balance** -> click the gray patch. Then adjust the black point by picking the
   black patch.
6. The correction applies to the entire clip. To copy it to other clips in the timeline:
   select the corrected clip -> `Ctrl/Cmd+C` -> select remaining clips -> `Ctrl/Cmd+V`.
   Or: **Color** menu -> **Copy Grade** -> **All Clips in Current Timeline**.
7. **Export:** switch to the **Deliver** page, choose your format and codec, and render.

![DaVinci Resolve: Color page, Auto Balance eyedropper on the gray patch](images/davinci_01_auto_balance.png)
*Screenshot placeholder — Color page, eyedropper picking the neutral gray patch.*

![DaVinci Resolve: Copy Grade to all clips](images/davinci_02_copy_grade.png)
*Screenshot placeholder — copying the grade to every clip in the timeline before rendering.*


## Video — Final Cut Pro

1. Import the video into a library. Place it in the timeline.
2. Select the clip. Open the **Color Inspector**: `Window -> Color Inspector` (or press `Cmd+6`).
3. Click **Add Color Correction** -> **Color Wheels** (or **Color Curves**).
4. In the viewer, scrub to the reference-target frame.
5. Click **Modify -> White Balance** in the menu bar, then click the neutral gray patch in
   the viewer. Final Cut shifts all three channels to neutralize that point.
6. To copy the correction to other clips: select the corrected clip -> `Cmd+C` -> select the
   remaining clips -> `Edit -> Paste Effects`.
7. To apply to every clip at once: select all clips (`Cmd+A`) -> paste effects.

![Final Cut Pro: Color Inspector, White Balance on the gray patch](images/fcp_01_white_balance.png)
*Screenshot placeholder — Color Inspector with White Balance picked on the gray patch.*


## Video — ffmpeg (command line)

ffmpeg can apply color corrections to a video in a single pass without an editing GUI.
The most practical approach is to compute per-channel gains or a Color Correction Matrix with
the Python code below and pass them directly to ffmpeg's `colorchannelmixer` filter.

**Simple white balance — per-channel gain:**

```bash
# Replace the gains with values printed by the white-balance cell below
ffmpeg -i input.mp4 \
  -vf "colorchannelmixer=rr=0.94:gg=1.0:bb=1.12" \
  -c:v libx264 -crf 18 -preset fast \
  output_corrected.mp4
```

**Full 3x3 color correction matrix (CCM):**

```bash
# colorchannelmixer accepts all 9 matrix entries
# rr rg rb = how much R, G, B contribute to the output Red channel, etc.
ffmpeg -i input.mp4 \
  -vf "colorchannelmixer=rr=0.95:rg=0.03:rb=0.02:gr=0.01:gg=1.01:gb=-0.02:br=-0.01:bg=0.04:bb=1.08" \
  -c:v libx264 -crf 18 -preset fast \
  output_corrected.mp4
```

**Batch-process a folder of videos:**

```bash
for f in ./recordings/*.mp4; do
  ffmpeg -i "$f" \
    -vf "colorchannelmixer=rr=0.94:gg=1.0:bb=1.12" \
    -c:v libx264 -crf 18 -preset fast \
    "./corrected/$(basename "$f")"
done
```


## Python / OpenCV

The rest of this notebook computes the correction programmatically and applies it to a video.
Two methods are shown, in order of increasing accuracy:

1. **White balance** — scales each channel independently so a neutral gray patch reads equal
   R, G, B. Fast; removes overall color casts (warm artificial light, underwater green shift).
2. **Color Correction Matrix (CCM)** — fits a 3x3 linear transform using all 24 ColorChecker
   patches. More accurate; corrects per-channel non-linearities.

The notebook uses a **synthetic ColorChecker image** with an artificial warm color cast so it
runs standalone. Replace it with your own ColorChecker photograph (or a frame grabbed from
your video) to calibrate real data.

**Requirements:** `pip install opencv-python numpy matplotlib`


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

### ColorChecker reference values

The ColorChecker Classic has 24 patches with known sRGB values (D50 illuminant). These are the
target our corrected images should match. The neutral gray patches (19–24, the bottom row) are
the most important for white balance, since they should read equal R = G = B when calibrated.


In [ ]:
# ColorChecker Classic — sRGB reference values (D50, 0-255)
# Row-major order matching the physical 4-row × 6-column layout.
COLORCHECKER_REF = np.array([
    [115,  82,  68],   #  1  Dark skin
    [194, 150, 130],   #  2  Light skin
    [ 98, 122, 157],   #  3  Blue sky
    [ 87, 108,  67],   #  4  Foliage
    [133, 128, 177],   #  5  Blue flower
    [103, 189, 170],   #  6  Bluish green
    [214, 126,  44],   #  7  Orange
    [ 80,  91, 166],   #  8  Purplish blue
    [193,  90,  99],   #  9  Moderate red
    [ 94,  60, 108],   # 10  Purple
    [157, 188,  64],   # 11  Yellow green
    [224, 163,  46],   # 12  Orange yellow
    [ 56,  61, 150],   # 13  Blue
    [ 70, 148,  73],   # 14  Green
    [175,  54,  60],   # 15  Red
    [231, 199,  31],   # 16  Yellow
    [187,  86, 149],   # 17  Magenta
    [  8, 133, 161],   # 18  Cyan
    [243, 243, 242],   # 19  White
    [200, 200, 200],   # 20  Neutral 8
    [160, 160, 160],   # 21  Neutral 6.5
    [122, 122, 121],   # 22  Neutral 5   <- used for white balance
    [ 85,  85,  85],   # 23  Neutral 3.5
    [ 52,  52,  52],   # 24  Black
], dtype=np.float32)

print(f"Loaded {len(COLORCHECKER_REF)} ColorChecker patches.")
print(f"Neutral patch (index 21, Neutral 5): RGB = {COLORCHECKER_REF[21]}")

### Create a synthetic test image

We generate a 4x6 grid of patches matching the ColorChecker layout, then apply an artificial
**warm color cast** (elevated red, reduced blue) to simulate images recorded under incandescent
light or at shallow depth underwater. Replace `img_cast` with your own photograph or video
frame to calibrate real data.


In [ ]:
N_COLS, N_ROWS = 6, 4
PATCH_PX  = 80   # patch size in pixels
BORDER_PX =  6   # border between patches


def make_synthetic_colorchecker(color_cast=(1.0, 1.0, 1.0)):
    """Return a BGR image of a synthetic ColorChecker with an optional color cast."""
    W = N_COLS * PATCH_PX + (N_COLS + 1) * BORDER_PX
    H = N_ROWS * PATCH_PX + (N_ROWS + 1) * BORDER_PX
    img = np.zeros((H, W, 3), dtype=np.float32)

    cast = np.array(color_cast, dtype=np.float32)

    for i, ref_rgb in enumerate(COLORCHECKER_REF):
        row, col = divmod(i, N_COLS)
        y = BORDER_PX + row * (PATCH_PX + BORDER_PX)
        x = BORDER_PX + col * (PATCH_PX + BORDER_PX)
        bgr = np.clip(ref_rgb * cast, 0, 255)[::-1]
        img[y:y + PATCH_PX, x:x + PATCH_PX] = bgr

    return img.astype(np.uint8)


def sample_patch(img_bgr, patch_idx, margin_frac=0.15):
    """Return the mean BGR value of the interior of a ColorChecker patch."""
    row, col = divmod(patch_idx, N_COLS)
    y0 = BORDER_PX + row * (PATCH_PX + BORDER_PX)
    x0 = BORDER_PX + col * (PATCH_PX + BORDER_PX)
    m = max(1, int(PATCH_PX * margin_frac))
    region = img_bgr[y0 + m:y0 + PATCH_PX - m, x0 + m:x0 + PATCH_PX - m]
    return region.mean(axis=(0, 1)).astype(np.float32)


# Warm cast: red boosted ×1.35, blue reduced ×0.72
# (mimics incandescent light or shallow underwater haze)
img_cast = make_synthetic_colorchecker(color_cast=(1.35, 1.0, 0.72))
img_ref  = make_synthetic_colorchecker(color_cast=(1.0,  1.0, 1.0))

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].imshow(cv2.cvtColor(img_ref,  cv2.COLOR_BGR2RGB))
axs[0].set_title("Reference (no cast)")
axs[0].axis("off")
axs[1].imshow(cv2.cvtColor(img_cast, cv2.COLOR_BGR2RGB))
axs[1].set_title("With warm color cast (R×1.35, B×0.72)")
axs[1].axis("off")
plt.suptitle("Synthetic ColorChecker — before correction", fontsize=13)
plt.tight_layout()
plt.show()

### Method 1 — White balance using the neutral gray patch

We sample patch 22 (Neutral 5, reference sRGB ~ 122, 122, 121) from the cast image.
Per-channel gain factors are computed so the sampled color matches the reference, then applied
to every pixel. This is equivalent to clicking the gray patch with the eyedropper in Lightroom
or DaVinci. The printed gains map directly to ffmpeg `colorchannelmixer=rr=..:gg=..:bb=..`.


In [ ]:
def compute_wb_gains(img_bgr, gray_patch_idx=21):
    """Per-channel gains to neutralize the chosen gray patch."""
    measured_bgr = sample_patch(img_bgr, gray_patch_idx)
    ref_rgb      = COLORCHECKER_REF[gray_patch_idx]
    ref_bgr      = ref_rgb[::-1]
    return ref_bgr / (measured_bgr + 1e-6)


def apply_gains(img_bgr, gains):
    out = img_bgr.astype(np.float32) * gains[np.newaxis, np.newaxis, :]
    return np.clip(out, 0, 255).astype(np.uint8)


wb_gains = compute_wb_gains(img_cast, gray_patch_idx=21)
img_wb   = apply_gains(img_cast, wb_gains)

print(f"White balance gains  B={wb_gains[0]:.3f}  G={wb_gains[1]:.3f}  R={wb_gains[2]:.3f}")
print(f"ffmpeg:  colorchannelmixer=rr={wb_gains[2]:.3f}:gg={wb_gains[1]:.3f}:bb={wb_gains[0]:.3f}")

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axs, [img_cast, img_wb, img_ref],
                           ["Cast", "After white balance", "Reference (ground truth)"]):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Method 2 — Color Correction Matrix (CCM)

A 3x3 CCM simultaneously maps all 24 measured patch colors toward their reference values using
**least-squares linear regression**:

$$\mathbf{M}_{\text{measured}} \cdot \mathbf{CCM} \approx \mathbf{R}_{\text{reference}}$$

where **M** is the 24x3 matrix of measured RGB values and **R** the 24x3 matrix of reference
values. This handles cross-channel contamination and is more accurate than white balance alone.


In [ ]:
def measure_all_patches(img_bgr):
    """Sample all 24 patches; return (24, 3) float32 array in RGB order."""
    return np.array([sample_patch(img_bgr, i)[::-1] for i in range(len(COLORCHECKER_REF))],
                    dtype=np.float32)


def compute_ccm(img_bgr):
    M = measure_all_patches(img_bgr)
    ccm, _, _, _ = np.linalg.lstsq(M, COLORCHECKER_REF, rcond=None)
    return ccm.astype(np.float32)


def apply_ccm(img_bgr, ccm):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    corrected_rgb = np.clip(img_rgb @ ccm, 0, 255).astype(np.uint8)
    return cv2.cvtColor(corrected_rgb, cv2.COLOR_RGB2BGR)


ccm     = compute_ccm(img_cast)
img_ccm = apply_ccm(img_cast, ccm)

print("Color Correction Matrix (maps measured RGB -> reference RGB):")
print(ccm.round(4))
print()
print("A perfect CCM for a linear color cast is diagonal.")
print("Off-diagonal terms correct cross-channel contamination.")

### Evaluate: per-patch error before and after correction

We compute the **mean absolute error (MAE)** in RGB space for each patch. Lower is better; an
MAE below ~5 (out of 255) is considered good for most ecological applications.


In [ ]:
def patch_errors(img_bgr):
    return np.abs(measure_all_patches(img_bgr) - COLORCHECKER_REF).mean(axis=1)


err_cast = patch_errors(img_cast)
err_wb   = patch_errors(img_wb)
err_ccm  = patch_errors(img_ccm)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, errs, title in zip(
    axs,
    [img_cast, img_wb, img_ccm],
    [err_cast, err_wb, err_ccm],
    ["Original (warm cast)", "White balance", "CCM corrected"],
):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title}\nMean error: {errs.mean():.1f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

patch_ids = np.arange(1, 25)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(patch_ids, err_cast, "o--", color="C1", label=f"Cast    mean={err_cast.mean():.1f}")
ax.plot(patch_ids, err_wb,   "s--", color="C2", label=f"WB      mean={err_wb.mean():.1f}")
ax.plot(patch_ids, err_ccm,  "^-",  color="C0", label=f"CCM     mean={err_ccm.mean():.1f}")
ax.axhline(5, color="gray", ls=":", lw=1, label="Error=5 threshold")
ax.set_xlabel("Patch number (1-24)")
ax.set_ylabel("Mean absolute error (RGB, 0-255)")
ax.set_title("Per-patch color error before and after correction")
ax.legend()
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

### Apply the correction to a video recording

The cell below reads an input video, derives the CCM from one reference frame where the
ColorChecker is visible, applies it to every frame, and writes a corrected copy.

Set `INPUT_VIDEO`, `OUTPUT_VIDEO`, and `REF_FRAME_IDX` to your own values. The cell raises a
clear error if the video can't be opened or the reference frame can't be read, so a
misconfiguration is obvious immediately.


In [ ]:
INPUT_VIDEO   = "recording.mp4"            # <- your input video
OUTPUT_VIDEO  = "recording_corrected.mp4"  # <- corrected output
REF_FRAME_IDX = 60                          # frame where the ColorChecker is visible

src = Path(INPUT_VIDEO)
if not src.is_file():
    raise FileNotFoundError(
        f"Input video not found: {src.resolve()}. "
        f"Set INPUT_VIDEO to a real recording, or use the synthetic example above."
    )

cap = cv2.VideoCapture(str(src))
if not cap.isOpened():
    raise RuntimeError(f"OpenCV could not open '{src}'. Is the codec installed?")

cap.set(cv2.CAP_PROP_POS_FRAMES, REF_FRAME_IDX)
ok, checker_frame = cap.read()
if not ok or checker_frame is None:
    cap.release()
    raise RuntimeError(
        f"Could not read reference frame {REF_FRAME_IDX} from '{src}'. "
        f"Pick a REF_FRAME_IDX within the clip where the ColorChecker is visible."
    )

# Derive the correction from the ColorChecker frame.
# Replace compute_ccm with compute_wb_gains/apply_gains for a quick white-balance fix.
ccm = compute_ccm(checker_frame)
print("CCM derived from reference frame:")
print(ccm.round(4))

fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open VideoWriter for '{OUTPUT_VIDEO}'.")

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
n = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    writer.write(apply_ccm(frame, ccm))
    n += 1
    if n % 200 == 0:
        print(f"  {n}/{total} frames")

cap.release()
writer.release()
print(f"Done — {n} frames written to {Path(OUTPUT_VIDEO).resolve()}")

## Tips

- **One reference shot per lighting condition**, not just per session. Re-film the target
  whenever the light changes (moving indoors/outdoors, time of day, cloud cover).
- **Avoid glare** on the reference target: tilt it slightly (less than 10 degrees) relative to
  the main light source.
- **Underwater:** red light is absorbed rapidly with depth, producing a strong blue-green cast.
  Take a reference shot at the start of each dive. For physically based underwater color
  restoration see [Sea-Thru](https://github.com/hainh/sea-thru) (Akkaynak & Treibitz 2019).
- **Export color space:** use sRGB for display/tracking pipelines, AdobeRGB or ProPhoto RGB
  for quantitative color analysis. Mixing color spaces across clips invalidates comparisons.
- **No re-encoding:** the gains and CCM printed above can be passed straight to ffmpeg's
  `colorchannelmixer` filter (see the ffmpeg section) to apply the correction in one pass.
